# Sensibilità MAE/RMSE per bin: MTGFlow e STGAN a t+1/t+6

Analisi post-hoc sugli stessi target SDE-Net e sul dominio comune dei due detector. MTGFlow varia il coefficiente $k$ di $Q3+k\,IQR$; STGAN varia la quota globale top-K. I dropout solari regionali isolati e l'ora di recupero sono esclusi prima di ricompattare la graduatoria STGAN. I riferimenti a priori sono rispettivamente $k=1.5$ e top 1%. Le curve sono prodotte sia globalmente sia separatamente negli stessi cinque bin percentuali di produzione dei report SDE-Net. Nessuna soglia viene scelta sul test 2019.

In [ ]:
from pathlib import Path
import json, os, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from physiq_pv.reporting import mtgflow_spatiotemporal as mtg_spatial
from physiq_pv.reporting.posthoc_outputs import PERCENT_PRODUCTION_BINS
from physiq_pv.reporting.detector_threshold_sensitivity import (
    join_detector_errors, load_mtgflow_coordinates, load_prediction_errors,
    load_stgan_coordinates, sensitivity_sweep, sensitivity_sweep_by_bin,
    plot_mae_dispersion,
)
from physiq_pv.reporting.pointwise_detector_posthoc import (
    detect_isolated_regional_solar_dropouts,
)

MTGFLOW_SEED_DIR = Path(os.environ.get('MTGFLOW_SEED_DIR', ROOT / 'outputs/pvgis_mtgflow/downstream_dense/seed_15')).resolve()
STGAN_SEED_DIR = Path(os.environ.get('STGAN_SEED_DIR', ROOT / 'outputs/pvgis_stgan/paper_reference/seed_20')).resolve()
SDE_RUN = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_h1-2-3-4-5-6_direct_seed1'
SDE_PREDICTIONS = Path(os.environ.get('SDE_MULTIHORIZON_PREDICTIONS', ROOT / 'outputs' / SDE_RUN / 'predictions.csv')).resolve()
TRAINING_STATS = Path(os.environ.get('MTGFLOW_TRAINING_STATS_CSV', ROOT / 'outputs/mtgflow_threshold_sensitivity/t_plus_1_and_6/training_iqr_by_location.csv')).resolve()
REFERENCE_PEAK_CSV = Path(os.environ.get(
    'SDE_REFERENCE_PEAK_CSV',
    ROOT / 'outputs' / SDE_RUN / 'posthoc_by_horizon' / 't_plus_1' / 'reference_production_peaks.csv',
)).resolve()
OUT_DIR = Path(os.environ.get('ANOMALY_SENSITIVITY_OUT_DIR', ROOT / 'outputs/anomaly_threshold_sensitivity_t1_t6')).resolve()
FIGURE_DIR = OUT_DIR / 'figures'; FIGURE_DIR.mkdir(parents=True, exist_ok=True)
HORIZONS = (1, 6)
MTGFLOW_K_VALUES = np.array([0.50, 1.00, 1.50, 2.00, 2.50, 3.00])
STGAN_TOP_PERCENT_VALUES = np.array([0.25, 0.50, 0.75, 1.00, 1.25, 1.50, 2.00, 3.00, 4.00, 5.00])
MTGFLOW_REFERENCE_K = 1.5
STGAN_REFERENCE_TOP_PERCENT = 1.0
DAYTIME_THRESHOLD_WM2 = 10.0
MIN_JOIN_COVERAGE = 0.95
PVGIS_2019_FILE = Path(os.environ.get(
    'PVGIS_2019_PATH', ROOT / 'data/pvgis/piedmont_pvgis_2019.nc'
)).resolve()
STGAN_PREPARED_MANIFEST = Path(os.environ.get(
    'STGAN_PREPARED_MANIFEST', ROOT / 'outputs/pvgis_stgan/prepared/manifest.csv'
)).resolve()
quality_override = os.environ.get('PVGIS_QUALITY_SOURCE')
PVGIS_QUALITY_SOURCE = (
    Path(quality_override).resolve() if quality_override
    else PVGIS_2019_FILE if PVGIS_2019_FILE.is_file()
    else STGAN_PREPARED_MANIFEST
)
print('SDE:', SDE_PREDICTIONS)
print('Reference peak:', REFERENCE_PEAK_CSV)
print('Quality source:', PVGIS_QUALITY_SOURCE)
print('Output:', OUT_DIR)

## 1. Errori SDE-Net sugli esatti timestamp target

In [ ]:
MTGFLOW_SCORES = MTGFLOW_SEED_DIR / 'anomaly_scores.csv'
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'
missing = [
    path for path in (MTGFLOW_SCORES, STGAN_SCORES, SDE_PREDICTIONS, REFERENCE_PEAK_CSV, PVGIS_QUALITY_SOURCE)
    if not path.is_file()
]
if missing: raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))
quality_issues = detect_isolated_regional_solar_dropouts(PVGIS_QUALITY_SOURCE)
QUALITY_TIMESTAMPS = pd.DatetimeIndex(pd.to_datetime(quality_issues['timestamp'])).drop_duplicates()
quality_issues.to_csv(OUT_DIR / 'pvgis_data_quality_issues.csv', index=False)
print(f'Esclusi {len(QUALITY_TIMESTAMPS)} timestamp: dropout regionali e recuperi immediati.')
reference_peak_table = pd.read_csv(REFERENCE_PEAK_CSV)
if 'reference_peak_w' not in reference_peak_table or reference_peak_table.empty:
    raise ValueError('reference_production_peaks.csv non contiene reference_peak_w.')
REFERENCE_PEAK_W = float(reference_peak_table['reference_peak_w'].iloc[0])
if not np.isfinite(REFERENCE_PEAK_W) or REFERENCE_PEAK_W <= 0:
    raise ValueError(f'Reference peak non valido: {REFERENCE_PEAK_W!r}')
errors = load_prediction_errors(
    SDE_PREDICTIONS, horizons=HORIZONS,
    daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
    reference_peak_w=REFERENCE_PEAK_W,
)
display(errors.groupby(['horizon_hours', 'production_bin'], observed=True).size().rename('n_daytime_before_common_domain'))

## 2. MTGFlow: sweep del coefficiente IQR $k$

In [ ]:
saved = mtg_spatial.read_saved_thresholds(MTGFLOW_SCORES)
threshold_table = mtg_spatial.build_threshold_table(
    saved, cached_statistics=TRAINING_STATS,
    per_site_training_paths=sorted(MTGFLOW_SEED_DIR.glob('*/train_scores.csv')),
    aggregate_training_path=MTGFLOW_SEED_DIR / 'train_anomaly_scores.csv',
)
mtg_coordinates = load_mtgflow_coordinates(
    MTGFLOW_SCORES, threshold_table, excluded_timestamps=QUALITY_TIMESTAMPS,
)
print('Coordinate MTGFlow valide:', len(mtg_coordinates))

## 3. STGAN: sweep della quota globale top-K

In [ ]:
stgan_coordinates = load_stgan_coordinates(
    STGAN_SCORES, excluded_timestamps=QUALITY_TIMESTAMPS,
    recompute_global_ranking=True,
)
common_keys = mtg_coordinates[['location', 'timestamp']].merge(
    stgan_coordinates[['location', 'timestamp']],
    on=['location', 'timestamp'], how='inner', validate='one_to_one',
)
common_errors = errors.merge(
    common_keys, on=['location', 'timestamp'], how='inner', validate='many_to_one',
)
COMMON_DOMAIN_COVERAGE = len(common_errors) / len(errors)
if COMMON_DOMAIN_COVERAGE < MIN_JOIN_COVERAGE:
    raise ValueError(f'Dominio comune detector/SDE insufficiente: {COMMON_DOMAIN_COVERAGE:.2%}')
mtg_joined = join_detector_errors(common_errors, mtg_coordinates, min_match_fraction=1.0)
stgan_joined = join_detector_errors(common_errors, stgan_coordinates, min_match_fraction=1.0)
mtg_sweep = sensitivity_sweep(
    mtg_joined, MTGFLOW_K_VALUES, detector='mtgflow',
    rare_when='coordinate_ge_threshold',
)
mtg_bin_sweep = sensitivity_sweep_by_bin(
    mtg_joined, MTGFLOW_K_VALUES, detector='mtgflow',
    rare_when='coordinate_ge_threshold',
)
stgan_sweep = sensitivity_sweep(
    stgan_joined, STGAN_TOP_PERCENT_VALUES, detector='stgan',
    rare_when='coordinate_le_threshold',
)
stgan_bin_sweep = sensitivity_sweep_by_bin(
    stgan_joined, STGAN_TOP_PERCENT_VALUES, detector='stgan',
    rare_when='coordinate_le_threshold',
)
stgan_sweep['threshold_kind'] = 'global_top_percent'
stgan_bin_sweep['threshold_kind'] = 'global_top_percent'
mtg_sweep['threshold_kind'] = 'iqr_k'
mtg_bin_sweep['threshold_kind'] = 'iqr_k'
print('Copertura dominio comune:', COMMON_DOMAIN_COVERAGE)
display(mtg_sweep[np.isclose(mtg_sweep['decision_threshold'], MTGFLOW_REFERENCE_K)])
display(stgan_sweep[np.isclose(stgan_sweep['decision_threshold'], STGAN_REFERENCE_TOP_PERCENT)])

## 4. MAE/RMSE e numerosità dei gruppi


Le linee MAE restano le medie degli errori assoluti su **tutti i campioni** del
gruppo a ciascuna soglia. La fascia scura contiene il 25°–75° percentile degli
errori assoluti; la fascia chiara arriva ai baffi Tukey (valori osservati entro
1,5 IQR dai quartili). Gli outlier non sono disegnati, ma contribuiscono al MAE.
Le fasce descrivono dispersione degli errori individuali: **non sono intervalli
di confidenza del MAE né bande predittive gaussiane**. La linea è la media,
non la mediana del boxplot, e può anche uscire dalla fascia. Nessuna aggregazione
giornaliera viene introdotta. I grafici RMSE restano invariati.


In [ ]:
def plot_detector_sweep(frame, detector, reference, xlabel):
    fig, axes = plt.subplots(len(HORIZONS), 2, figsize=(14, 5 * len(HORIZONS)), squeeze=False)
    for row, horizon in enumerate(HORIZONS):
        data = frame[frame['horizon_hours'].eq(horizon)]
        for column, metric in enumerate(('mae', 'rmse')):
            axis = axes[row, column]
            if metric == 'mae':
                plot_mae_dispersion(axis, data)
            else:
                axis.plot(data['decision_threshold'], data[f'{metric}_normal'], marker='o', label='Normali')
                axis.plot(data['decision_threshold'], data[f'{metric}_rare'], marker='o', label='Rari/anomali')
            axis.axvline(reference, color='black', linestyle='--', label='Riferimento a priori')
            axis.set(title=f'{detector.upper()} — {metric.upper()} t+{horizon}', xlabel=xlabel, ylabel=f'{metric.upper()} [W]')
            axis.grid(alpha=.25); axis.legend()
    fig.tight_layout(); path = FIGURE_DIR / f'{detector}_mae_rmse_sensitivity_t1_t6.png'
    fig.savefig(path, dpi=180, bbox_inches='tight'); plt.show(); return path

metric_paths = [
    plot_detector_sweep(mtg_sweep, 'mtgflow', MTGFLOW_REFERENCE_K, 'Coefficiente IQR k'),
    plot_detector_sweep(stgan_sweep, 'stgan', STGAN_REFERENCE_TOP_PERCENT, 'Quota globale top-K [%]'),
]

def plot_detector_sweep_by_bin(frame, detector, reference, xlabel):
    paths = {}
    ordered_bins = [name for name, _, _ in PERCENT_PRODUCTION_BINS]
    for production_bin in ordered_bins:
        bin_data = frame[frame['production_bin'].eq(production_bin)]
        if bin_data.empty:
            continue
        fig, axes = plt.subplots(
            len(HORIZONS), 3, figsize=(19, 5 * len(HORIZONS)), squeeze=False
        )
        for row, horizon in enumerate(HORIZONS):
            data = bin_data[bin_data['horizon_hours'].eq(horizon)]
            if data.empty:
                raise ValueError(f'Bin {production_bin} privo di t+{horizon}.')
            for column, metric in enumerate(('mae', 'rmse')):
                axis = axes[row, column]
                if metric == 'mae':
                    plot_mae_dispersion(axis, data)
                else:
                    axis.plot(data['decision_threshold'], data[f'{metric}_normal'], marker='o', label='Normali')
                    axis.plot(data['decision_threshold'], data[f'{metric}_rare'], marker='o', label='Rari/anomali')
                axis.axvline(reference, color='black', linestyle='--', label='Riferimento a priori')
                axis.set(
                    title=f'{detector.upper()} — {metric.upper()} t+{horizon}',
                    xlabel=xlabel, ylabel=f'{metric.upper()} [W]',
                )
                axis.grid(alpha=.25); axis.legend()
            class_axis = axes[row, 2]
            class_axis.plot(
                data['decision_threshold'], 100 * data['rare_fraction'],
                marker='o', color='tab:red',
            )
            class_axis.axvline(reference, color='black', linestyle='--')
            class_axis.set(
                title=f'{detector.upper()} — campioni anomali t+{horizon}',
                xlabel=xlabel, ylabel='Campioni anomali [%]',
            )
            class_axis.grid(alpha=.25)
        fig.suptitle(f'Sensibilità threshold — {production_bin}', y=1.002)
        fig.tight_layout()
        path = FIGURE_DIR / f'{detector}_threshold_sensitivity_{production_bin}_t1_t6.png'
        fig.savefig(path, dpi=180, bbox_inches='tight')
        plt.show()
        paths[production_bin] = path
    return paths

bin_metric_paths = {
    'mtgflow': plot_detector_sweep_by_bin(
        mtg_bin_sweep, 'mtgflow', MTGFLOW_REFERENCE_K, 'Coefficiente IQR k'
    ),
    'stgan': plot_detector_sweep_by_bin(
        stgan_bin_sweep, 'stgan', STGAN_REFERENCE_TOP_PERCENT, 'Quota globale top-K [%]'
    ),
}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, frame, detector, reference, xlabel in (
    (axes[0], mtg_sweep, 'MTGFlow', MTGFLOW_REFERENCE_K, 'Coefficiente IQR k'),
    (axes[1], stgan_sweep, 'STGAN', STGAN_REFERENCE_TOP_PERCENT, 'Quota top-K [%]'),
):
    for horizon in HORIZONS:
        data = frame[frame['horizon_hours'].eq(horizon)]
        axis.plot(data['decision_threshold'], 100 * data['rare_fraction'], marker='o', label=f't+{horizon}')
    axis.axvline(reference, color='black', linestyle='--'); axis.set(title=f'{detector}: classificazione', xlabel=xlabel, ylabel='Campioni anomali [%]'); axis.grid(alpha=.25); axis.legend()
fig.tight_layout(); classification_path = FIGURE_DIR / 'classification_sensitivity_mtgflow_stgan_t1_t6.png'
fig.savefig(classification_path, dpi=180, bbox_inches='tight'); plt.show()


## 5. Sovrapposizione dei detector ai riferimenti a priori

Il confronto usa una sola copia di ogni coordinata diurna nel dominio comune. Riporta intersezione, Jaccard, quota di STGAN catturata da MTGFlow, quota di MTGFlow catturata da STGAN e correlazione giornaliera.

In [ ]:
reference_domain = common_errors.loc[
    common_errors['horizon_hours'].eq(HORIZONS[0]), ['location', 'timestamp']
].drop_duplicates()
reference_flags = (
    reference_domain
    .merge(
        mtg_coordinates, on=['location', 'timestamp'], how='left',
        validate='one_to_one',
    )
    .rename(columns={'decision_coordinate': 'mtgflow_k_coordinate'})
    .merge(
        stgan_coordinates, on=['location', 'timestamp'], how='left',
        validate='one_to_one',
    )
    .rename(columns={'decision_coordinate': 'stgan_top_percent_coordinate'})
)
if reference_flags[['mtgflow_k_coordinate', 'stgan_top_percent_coordinate']].isna().any().any():
    raise ValueError('Il dominio comune contiene coordinate detector mancanti.')
reference_flags['mtgflow_rare'] = reference_flags['mtgflow_k_coordinate'].ge(MTGFLOW_REFERENCE_K)
reference_flags['stgan_rare'] = reference_flags['stgan_top_percent_coordinate'].le(STGAN_REFERENCE_TOP_PERCENT)
reference_flags['both_rare'] = reference_flags['mtgflow_rare'] & reference_flags['stgan_rare']
n_total = len(reference_flags)
n_mtgflow = int(reference_flags['mtgflow_rare'].sum())
n_stgan = int(reference_flags['stgan_rare'].sum())
n_both = int(reference_flags['both_rare'].sum())
n_union = int((reference_flags['mtgflow_rare'] | reference_flags['stgan_rare']).sum())
reference_overlap = pd.DataFrame([{'n_common_daytime': n_total, 'n_mtgflow_rare': n_mtgflow, 'n_stgan_rare': n_stgan, 'n_both_rare': n_both, 'jaccard': n_both / n_union if n_union else np.nan, 'stgan_captured_by_mtgflow': n_both / n_stgan if n_stgan else np.nan, 'mtgflow_captured_by_stgan': n_both / n_mtgflow if n_mtgflow else np.nan}])
daily_overlap = reference_flags.assign(
    day=reference_flags['timestamp'].dt.normalize(),
).groupby('day', as_index=False).agg(
    mtgflow_anomalies=('mtgflow_rare', 'sum'),
    stgan_anomalies=('stgan_rare', 'sum'),
    shared_anomalies=('both_rare', 'sum'),
)
reference_overlap['daily_pearson'] = daily_overlap['mtgflow_anomalies'].corr(daily_overlap['stgan_anomalies'], method='pearson')
reference_overlap['daily_spearman'] = daily_overlap['mtgflow_anomalies'].corr(daily_overlap['stgan_anomalies'], method='spearman')
reference_overlap.to_csv(OUT_DIR / 'reference_detector_overlap.csv', index=False)
daily_overlap.to_csv(OUT_DIR / 'reference_detector_daily_overlap.csv', index=False)
display(reference_overlap)
display(daily_overlap.sort_values(['shared_anomalies', 'stgan_anomalies', 'mtgflow_anomalies'], ascending=False).head(20))
del mtg_joined, stgan_joined, reference_flags

## 6. Riferimenti e selezione senza leakage

Le curve 2019 descrivono la sensibilità, ma non devono essere usate per scegliere e poi dichiarare ottimale una soglia sullo stesso test. La scelta finale va effettuata su validation e congelata. In assenza di tale selezione restano validi i riferimenti del protocollo: MTGFlow $k=1.5$ e STGAN top 1%.

In [ ]:
sweep = pd.concat([mtg_sweep, stgan_sweep], ignore_index=True)
sweep.to_csv(OUT_DIR / 'detector_threshold_sensitivity_metrics.csv', index=False)
bin_sweep = pd.concat([mtg_bin_sweep, stgan_bin_sweep], ignore_index=True)
bin_sweep.to_csv(OUT_DIR / 'detector_threshold_sensitivity_by_bin_metrics.csv', index=False)
references = sweep[(sweep['detector'].eq('mtgflow') & np.isclose(sweep['decision_threshold'], MTGFLOW_REFERENCE_K)) | (sweep['detector'].eq('stgan') & np.isclose(sweep['decision_threshold'], STGAN_REFERENCE_TOP_PERCENT))].copy()
references.to_csv(OUT_DIR / 'reference_decision_metrics.csv', index=False)
bin_references = bin_sweep[(bin_sweep['detector'].eq('mtgflow') & np.isclose(bin_sweep['decision_threshold'], MTGFLOW_REFERENCE_K)) | (bin_sweep['detector'].eq('stgan') & np.isclose(bin_sweep['decision_threshold'], STGAN_REFERENCE_TOP_PERCENT))].copy()
bin_references.to_csv(OUT_DIR / 'reference_decision_by_bin_metrics.csv', index=False)
threshold_table.to_csv(OUT_DIR / 'mtgflow_training_thresholds_by_location.csv', index=False)
metadata = {
    'post_processing_only': True, 'training_rerun': False,
    'mae_bands': 'absolute-error Q1-Q3 and observed Tukey whiskers; not confidence intervals',
    'mae_line': 'pooled mean absolute error, including outliers',
    'horizons_hours': list(HORIZONS), 'daytime_threshold_wm2': DAYTIME_THRESHOLD_WM2,
    'mtgflow_rule': 'score >= Q3 + k * IQR', 'mtgflow_reference_k': MTGFLOW_REFERENCE_K,
    'stgan_rule': 'global top-K by score percentile', 'stgan_reference_top_percent': STGAN_REFERENCE_TOP_PERCENT,
    'reference_peak_w': REFERENCE_PEAK_W, 'reference_peak_csv': str(REFERENCE_PEAK_CSV),
    'production_bin_basis': '100 * y_true / fixed training reference peak',
    'production_bins': [name for name, _, _ in PERCENT_PRODUCTION_BINS],
    'quality_source': str(PVGIS_QUALITY_SOURCE),
    'quality_filter_policy': 'isolated_regional_solar_dropout_plus_immediate_recovery',
    'excluded_quality_timestamps': len(QUALITY_TIMESTAMPS),
    'stgan_ranking': 'global ranking recomputed after quality exclusion',
    'comparison_domain': 'common MTGFlow/STGAN daytime coordinates',
    'common_domain_coverage': COMMON_DOMAIN_COVERAGE,
    'selection_policy': 'select_on_validation_then_freeze; 2019 is test-only',
}
(OUT_DIR / 'analysis_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
display(references); display(bin_references); print('Output:', OUT_DIR)

## 7. Dati da copiare per analizzare il comportamento delle soglie

Eseguire la cella e inviare tutto il testo compreso tra `BEGIN_THRESHOLD_BEHAVIOR_CSV` e `END_THRESHOLD_BEHAVIOR_CSV`. Il report contiene risultati globali e per bin, per entrambi i detector e per t+1/t+6.

In [ ]:
def make_copy_report(frame, scope):
    report = frame.copy()
    report['scope'] = scope
    if 'production_bin' not in report:
        report['production_bin'] = 'all_daytime'
    report['n_total'] = report['n_normal'] + report['n_rare']
    report['anomaly_pct'] = 100.0 * report['rare_fraction']
    report['mae_gap_rare_minus_normal'] = report['mae_rare'] - report['mae_normal']
    report['rmse_gap_rare_minus_normal'] = report['rmse_rare'] - report['rmse_normal']
    report['threshold_meaning'] = np.where(
        report['detector'].eq('mtgflow'),
        'larger_k_more_selective',
        'larger_top_percent_more_permissive',
    )
    columns = [
        'scope', 'production_bin', 'detector', 'horizon_hours',
        'threshold_kind', 'decision_threshold', 'threshold_meaning',
        'n_total', 'n_normal', 'n_rare', 'anomaly_pct',
        'mae_normal', 'mae_rare', 'mae_gap_rare_minus_normal',
        'rmse_normal', 'rmse_rare', 'rmse_gap_rare_minus_normal',
    ]
    return report[columns]

copy_report = pd.concat([
    make_copy_report(sweep, 'overall'),
    make_copy_report(bin_sweep, 'production_bin'),
], ignore_index=True)
copy_report = copy_report.sort_values(
    ['scope', 'production_bin', 'detector', 'horizon_hours', 'decision_threshold']
).reset_index(drop=True)
copy_report_path = OUT_DIR / 'threshold_behavior_copy_report.csv'
copy_report.to_csv(copy_report_path, index=False)
copy_csv = copy_report.to_csv(
    index=False, float_format='%.6g', lineterminator='\n'
)
print('BEGIN_THRESHOLD_BEHAVIOR_CSV')
print(copy_csv, end='')
print('END_THRESHOLD_BEHAVIOR_CSV')
print('File:', copy_report_path)